#1. 리뷰데이터 크롤링

In [ ]:
!pip install google-play-scraper
!pip install pandas openpyxl
!pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.1 MB/s eta 0:00:00


In [ ]:
from google_play_scraper import reviews_all, Sort
import pandas as pd
from datetime import datetime
import os
from shutil import move
from IPython.display import display

In [ ]:
APP_ID = "co.benx.weverse"
COUNTRY = "us"
TARGET_COUNT = 5000000  # 최종 목표 리뷰 수
EXCEL_FILE = "playstore_reviews.xlsx"

# ✅ 날짜 필터 ON/OFF 설정
USE_DATE_FILTER = False  # True → 날짜 필터 적용, False → 적용 안 함
DATE_FILTER = "2025-4-08"  # 적용할 날짜

if USE_DATE_FILTER:
    filter_date = datetime.strptime(DATE_FILTER, "%Y-%m-%d")
    print(f"[INFO] {DATE_FILTER} 이후 작성된 리뷰만 수집합니다.")
else:
    print(f"[INFO] 날짜 필터 사용하지 않습니다. 전체 리뷰 사용합니다.")

[INFO] 날짜 필터 사용하지 않습니다. 전체 리뷰 사용합니다.


In [ ]:
from tqdm import tqdm
from google_play_scraper import reviews, Sort
import pandas as pd

print(f"[INFO] 전체 리뷰 수집 시작...")

all_reviews = []
page = 1
batch_size = 100
token = None

with tqdm(total=TARGET_COUNT, desc="리뷰 수집 진행중", unit="개") as pbar:
    while len(all_reviews) < TARGET_COUNT:
        result, continuation = reviews(
            APP_ID,
            lang=COUNTRY,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=batch_size,
            continuation_token=token
        )

        if not result:
            break

        for review in result:
            review_date = review['at']
            if USE_DATE_FILTER:
                if review_date >= filter_date:
                    all_reviews.append(review)
            else:
                all_reviews.append(review)

        token = continuation
        pbar.update(len(result))
        page += 1

print(f"[INFO] 리뷰 수집 완료! 총 {len(all_reviews)}개")

# ==========================
# 📦 DataFrame 변환
# ==========================
df = pd.DataFrame(all_reviews)

# ==========================
# 🔁 중복 제거 (userName + at 기준)
# ==========================
before = len(df)
df.drop_duplicates(subset=["userName", "at"], inplace=True)
after = len(df)
print(f"[INFO] 중복 제거 완료: {before} → {after}개")

# ==========================
# 🧹 필요 없는 컬럼 제거
# ==========================
columns_to_drop = ["userImage", "reviewCreatedVersion", "replyContent", "repliedAt", "appVersion"]
existing_columns = [col for col in columns_to_drop if col in df.columns]
df.drop(columns=existing_columns, inplace=True)
print(f"[INFO] 필요 없는 컬럼 제거 완료: {existing_columns}")

# ==========================
# 🎯 목표 개수 맞추기
# ==========================
if after >= TARGET_COUNT:
    df = df.iloc[:TARGET_COUNT]
    print("리뷰 크롤링 완료!")
else:
    print(f"[⚠️ WARNING] 목표 개수에 도달하지 못했습니다. 최종 {after}개 리뷰만 확보.")

rename_map = {
    "content": "comment",
    "score": "score",
    "at": "date",
    "userName": "username",
    "thumbsUpCount": "likes count"
}
df.rename(columns=rename_map, inplace=True)

df.to_excel(EXCEL_FILE, index=False)
print(f"[✅ 저장 완료] '{EXCEL_FILE}'에 최종 {len(df)}개 리뷰 저장됨!")

from google.colab import files
files.download(EXCEL_FILE)


[INFO] 전체 리뷰 수집 시작...


리뷰 수집 진행중:   2%|▏         | 99009/5000000 [07:21<6:04:06, 224.33개/s]


[INFO] 리뷰 수집 완료! 총 99009개
[INFO] 중복 제거 완료: 99009 → 98595개
[INFO] 필요 없는 컬럼 제거 완료: ['userImage', 'reviewCreatedVersion', 'replyContent', 'repliedAt', 'appVersion']
[⚠️ WARNING] 목표 개수에 도달하지 못했습니다. 최종 98595개 리뷰만 확보.


#2. 전처리(음절필터&이모지 제거)

In [ ]:
!pip uninstall -y pandas numpy
!pip install pandas numpy --upgrade --force-reinstall

Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 25.4 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2025.2

In [ ]:
!pip install nltk emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 10.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import emoji
from tqdm import tqdm
from google.colab import files

# 파일 업로드
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# tqdm과 pandas 연동
tqdm.pandas()

# 엑셀 파일 읽기
df = pd.read_excel(file_name)

# 이모지 + 비영어 문자 제거 함수
def clean_text(text):
    if isinstance(text, float):
        text = str(text)
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.strip()

# 전처리 적용
df['cleaned_comment'] = df['comment'].progress_apply(clean_text)

# x단어 이하 리뷰 제거
df['word_count'] = df['cleaned_comment'].apply(lambda x: len(x.split()))
df = df[df['word_count'] > 5].reset_index(drop=True)

# 🔄 열 이름 변경
df = df.rename(columns={
    'comment': 'previous comment',        # 원래 comment → previous comment
    'cleaned_comment': 'comment'          # 전처리된 comment → comment
})

# 👉 word_count 열은 불필요하면 제거
df = df.drop(columns=['word_count'])

# ✅ 엑셀로 저장
preprocessed_file_path = 'preprocessed_comments.xlsx'
df.to_excel(preprocessed_file_path, index=False)
print(f"✅ 전처리된 데이터를 '{preprocessed_file_path}' 파일로 저장했습니다.")

# 다운로드
files.download(preprocessed_file_path)


ModuleNotFoundError: No module named 'emoji'

#3. 랜덤 Sampling

In [ ]:
import pandas as pd
from google.colab import files

# 엑셀 파일 업로드
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 엑셀 파일 불러오기
df = pd.read_excel(file_name)

# 전체 개수 확인
print(f"전체 데이터 수: {len(df)}개")

# 샘플링 개수 입력
n = int(input("몇 개를 랜덤 샘플링할까요? 숫자를 입력하세요: "))

# 유효성 검사
if n > len(df):
    raise ValueError("❌ 샘플 수가 전체 데이터보다 많습니다.")

# 랜덤 샘플링
sampled_df = df.sample(n=n, random_state=42).reset_index(drop=True)

# 샘플링 결과 저장
sampled_file = f"random_sample_{n}.xlsx"
sampled_df.to_excel(sampled_file, index=False)

print(f"✅ {n}개의 랜덤 샘플 데이터를 '{sampled_file}' 파일로 저장했습니다.")
files.download(sampled_file)


Saving preprocessed_comments (2).xlsx to preprocessed_comments (2).xlsx
전체 데이터 수: 14980개
몇 개를 랜덤 샘플링할까요? 숫자를 입력하세요: 10000
✅ 10000개의 랜덤 샘플 데이터를 'random_sample_10000.xlsx' 파일로 저장했습니다.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install transformers pandas openpyxl tqdm matplotlib datasets scikit-learn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 11.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==1

#4. BIG 5 Personality 문장 관련성 측정(Zero-shot classification)

OPENNESS

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 📄 엑셀 데이터 불러오기
df = pd.read_excel(filename)

# ✅ 'comment' 열 존재 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 🎯 Openness 라벨 정의
openness_labels = [
    ("Very Conservative", 0.0),
    ("A bit Conservative", 0.25),
    ("Neutral", 0.5),
    ("A bit Open", 0.75),
    ("Very Open", 1.0)
]

# 🧠 Openness만 softmax 기반 점수 계산 함수
def classify_openness_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in openness_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="The personality of the respondent is {} in terms of Big Five Factors.",
        return_all_scores=True
    )

    # output은 list of dicts 형태여야 함
    if isinstance(output, list):
        result = output
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, openness_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Openness_score"] = df["comment"].progress_apply(classify_openness_score)

# 💾 결과 저장 및 다운로드
output_file = filename.replace(".xlsx", "_openness_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)


Saving Replika 샘플링 10000.xlsx to Replika 샘플링 10000.xlsx


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0
100%|██████████| 10000/10000 [21:50<00:00,  7.63it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CONSCIENTIOUSNESS

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 📄 엑셀 데이터 불러오기
df = pd.read_excel(filename)

# ✅ 'comment' 열 존재 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 🎯 Conscientiousness 라벨 정의
conscientiousness_labels = [
    ("Very Careless", 0.0),
    ("A bit Careless", 0.25),
    ("Neutral", 0.5),
    ("A bit Conscientious", 0.75),
    ("Very Conscientious", 1.0)
]

# 🧠 Conscientiousness 점수 계산 함수
def classify_conscientiousness_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in conscientiousness_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="The personality of the respondent is {} in terms of Big Five Factors.",
        return_all_scores=True
    )

    # output은 list of dicts 형태여야 함
    if isinstance(output, list):
        result = output
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, conscientiousness_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Conscientiousness_score"] = df["comment"].progress_apply(classify_conscientiousness_score)

# 💾 결과 저장 및 다운로드
output_file = filename.replace(".xlsx", "_conscientiousness_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)


Saving Replika 샘플링 10000_openness_scored.xlsx to Replika 샘플링 10000_openness_scored (1).xlsx


Device set to use cuda:0
100%|██████████| 10000/10000 [22:17<00:00,  7.48it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

EXTRAVERSION

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 📄 엑셀 데이터 불러오기
df = pd.read_excel(filename)

# ✅ 'comment' 열 존재 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 🎯 Extraversion 라벨 정의
extraversion_labels = [
    ("Very Introverted", 0.0),
    ("A bit Introverted", 0.25),
    ("Neutral", 0.5),
    ("A bit Extraverted", 0.75),
    ("Very Extraverted", 1.0)
]

# 🧠 Extraversion 점수 계산 함수
def classify_extraversion_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in extraversion_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="The personality of the respondent is {} in terms of Big Five Factors.",
        return_all_scores=True
    )

    # output은 list of dicts 형태여야 함
    if isinstance(output, list):
        result = output
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, extraversion_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Extraversion_score"] = df["comment"].progress_apply(classify_extraversion_score)

# 💾 결과 저장 및 다운로드
output_file = filename.replace(".xlsx", "_extraversion_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)


Saving Replika 샘플링 10000_openness_scored (1)_conscientiousness_scored.xlsx to Replika 샘플링 10000_openness_scored (1)_conscientiousness_scored (1).xlsx


Device set to use cuda:0
100%|██████████| 10000/10000 [22:11<00:00,  7.51it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

AGREEABLENESS

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 📄 엑셀 데이터 불러오기
df = pd.read_excel(filename)

# ✅ 'comment' 열 존재 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 🎯 Agreeableness 라벨 정의
agreeableness_labels = [
    ("Very Antagonistic", 0.0),
    ("A bit Antagonistic", 0.25),
    ("Neutral", 0.5),
    ("A bit Agreeable", 0.75),
    ("Very Agreeable", 1.0)
]

# 🧠 Agreeableness 점수 계산 함수
def classify_agreeableness_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in agreeableness_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="The personality of the respondent is {} in terms of Big Five Factors.",
        return_all_scores=True
    )

    # output은 list of dicts 형태여야 함
    if isinstance(output, list):
        result = output
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, agreeableness_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Agreeableness_score"] = df["comment"].progress_apply(classify_agreeableness_score)

# 💾 결과 저장 및 다운로드
output_file = filename.replace(".xlsx", "_agreeableness_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)


Saving Replika 샘플링 10000_openness_scored (1)_conscientiousness_scored (1)_extraversion_scored.xlsx to Replika 샘플링 10000_openness_scored (1)_conscientiousness_scored (1)_extraversion_scored (1).xlsx


Device set to use cuda:0
100%|██████████| 10000/10000 [22:10<00:00,  7.51it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

NEUROTICISM

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 📄 엑셀 데이터 불러오기
df = pd.read_excel(filename)

# ✅ 'comment' 열 존재 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 🎯 Neuroticism 라벨 정의
neuroticism_labels = [
    ("Very Emotionally Stable", 0.0),
    ("A bit Emotionally Stable", 0.25),
    ("Neutral", 0.5),
    ("A bit Neurotic", 0.75),
    ("Very Neurotic", 1.0)
]

# 🧠 Neuroticism 점수 계산 함수
def classify_neuroticism_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in neuroticism_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="The personality of the respondent is {} in terms of Big Five Factors.",
        return_all_scores=True
    )

    # output은 list of dicts 형태여야 함
    if isinstance(output, list):
        result = output
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, neuroticism_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Neuroticism_score"] = df["comment"].progress_apply(classify_neuroticism_score)

# 💾 결과 저장 및 다운로드
output_file = filename.replace(".xlsx", "_neuroticism_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)


Saving Replika 샘플링 10000_openness_scored (1)_conscientiousness_scored (1)_extraversion_scored (1)_agreeableness_scored.xlsx to Replika 샘플링 10000_openness_scored (1)_conscientiousness_scored (1)_extraversion_scored (1)_agreeableness_scored (1).xlsx


Device set to use cpu
  0%|          | 4/10000 [00:11<7:43:39,  2.78s/it]


KeyboardInterrupt: 

#5. 인간 유사성 문장 관련성 측정(Zero-shot classification)

In [ ]:
!pip install scipy transformers pandas openpyxl tqdm matplotlib datasets scikit-learn --quiet

HUMANLIKENESS

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 📄 엑셀 데이터 불러오기
df = pd.read_excel(filename)

# ✅ 'comment' 열 존재 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 🎯 Humanlikeness 라벨 정의
humanlikeness_labels = [
    ("Very Machianlike", 0.0),
    ("Somewhat Machianlike", 0.25),
    ("Neutral", 0.5),
    ("Somewhat Humanlike", 0.75),
    ("Very Humanlike", 1.0)
]

# 🧠 Humanlikeness 점수 계산 함수
def classify_humanlikeness_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in humanlikeness_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="This comment mention to {} in terms of uncanny valley theory in conversational AI.",
        return_all_scores=True
    )

    # output 검증 및 가중 평균 계산
    if isinstance(output, list):
        result = output
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, humanlikeness_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Humanlikeness_score"] = df["comment"].progress_apply(classify_humanlikeness_score)

# 💾 결과 저장 및 다운로드
output_file = filename.replace(".xlsx", "_humanlikeness_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)

print("✅ 완료: Humanlikeness_score (0~1)가 계산되어 저장되었습니다.")


Saving 제로샷 돌리기용.xlsx to 제로샷 돌리기용.xlsx


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0
100%|██████████| 78069/78069 [2:42:04<00:00,  8.03it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ 완료: Humanlikeness_score (0~1)가 계산되어 저장되었습니다.


FAMILIARITY

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 📄 엑셀 데이터 불러오기
df = pd.read_excel(filename)

# ✅ 'comment' 열 존재 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 🎯 Familiarity 라벨 정의
familiarity_labels = [
    ("Very Unfamiliar", 0.0),
    ("Somewhat Unfamiliar", 0.25),
    ("Neutral", 0.5),
    ("Somewhat Familiar", 0.75),
    ("Very Familiar", 1.0)
]

# 🧠 Familiarity 점수 계산 함수
def classify_familiarity_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in familiarity_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="This comment refers to {} in terms of uncanny valley theory in conversational AI.",
        return_all_scores=True
    )

    # output 검증 및 가중 평균 계산
    if isinstance(output, list):
        result = output
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, familiarity_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Familiarity_score"] = df["comment"].progress_apply(classify_familiarity_score)

# 💾 결과 저장 및 다운로드
output_file = filename.replace(".xlsx", "_familiarity_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)

print("✅ 완료: Familiarity_score (0~1)가 계산되어 저장되었습니다.")


Saving GPT 샘플링 10000_BIG5,주관,양극,humanlikeness.xlsx to GPT 샘플링 10000_BIG5,주관,양극,humanlikeness.xlsx


Device set to use cuda:0
100%|██████████| 10000/10000 [19:16<00:00,  8.65it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ 완료: Familiarity_score (0~1)가 계산되어 저장되었습니다.


#6. 주관성 문장 관련성 측정(Zero-shot classification)

SUBJECTIVITY

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 📄 엑셀 데이터 불러오기
df = pd.read_excel(filename)

# ✅ 'comment' 열 존재 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 🎯 Subjectivity 라벨 정의 (5단계)
subjectivity_labels = [
    ("Very Objective", 0.0),
    ("Somewhat Objective", 0.25),
    ("Neutral", 0.5),
    ("Somewhat Subjective", 0.75),
    ("Very Subjective", 1.0)
]

# 🧠 Subjectivity 점수 계산 함수
def classify_subjectivity_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in subjectivity_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="This sentence expresses a {} opinion.",
        return_all_scores=True
    )

    # output 검증 및 가중 평균 계산
    if isinstance(output, list):
        result = output
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, subjectivity_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Subjectivity_score"] = df["comment"].progress_apply(classify_subjectivity_score)

# 💾 결과 저장 및 다운로드
output_file = filename.replace(".xlsx", "_subjectivity_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)

print("✅ 완료: Subjectivity_score (0~1)가 계산되어 저장되었습니다.")


Saving GPT 샘플링 10000_BIG5.xlsx to GPT 샘플링 10000_BIG5.xlsx


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0
100%|██████████| 10000/10000 [18:18<00:00,  9.11it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ 완료: Subjectivity_score (0~1)가 계산되어 저장되었습니다.


#7. 리뷰 길이 측정

In [ ]:
# ✅ 설치
!pip install pandas openpyxl textblob nltk --quiet

# 📚 라이브러리 로딩
import pandas as pd
import numpy as np
from textblob import TextBlob
import nltk
from tqdm import tqdm
from google.colab import files

# 📥 초기 다운로드 (한 번만)
nltk.download('punkt')

# 📂 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 📄 데이터 로드
df = pd.read_excel(filename)
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 존재하지 않습니다.")
df = df[['comment']].dropna().reset_index(drop=True)

# ✨ 감성 분석 함수 (전처리 제거 + vader 제외)
def analyze(text):
    try:
        blob = TextBlob(str(text))
        return {
            'review_length': float(len(str(text).split())),
        }
    except Exception:
        return {
            'review_length': 0.0,
        }

# 🔁 적용
tqdm.pandas()
sentiment_df = df['comment'].progress_apply(analyze).apply(pd.Series)

# 🔗 결과 결합
df_result = pd.concat([df, sentiment_df], axis=1)

# 💾 저장 및 다운로드
output_file = filename.replace(".xlsx", "_with_textblob_sentiment.xlsx")
df_result.to_excel(output_file, index=False)
files.download(output_file)

print("✅ 리뷰 길이 분석 완료!")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Saving random_sample_10000_zero_shot_bigfive.xlsx to random_sample_10000_zero_shot_bigfive.xlsx


100%|██████████| 10000/10000 [00:03<00:00, 3250.18it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ 리뷰 감성 분석 완료!


#8. 감성 분석(RoBERTa)

In [ ]:
!pip install transformers torch pandas openpyxl tqdm scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 891.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [ ]:
# 📚 라이브러리
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm import tqdm
from google.colab import files

# ✅ 디바이스 설정 (GPU 있으면 사용)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 📂 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 📄 데이터 로딩
df = pd.read_excel(filename)
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 없습니다.")
df['comment'] = df['comment'].fillna("")

# 🤖 모델 로드
MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(device)

# ✅ 감성 분석 함수 (batch 처리)
def analyze_sentiment_batch(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    probs = softmax(outputs.logits.cpu().numpy(), axis=1)
    return probs  # shape: (batch_size, 3)

# ✅ 배치 감성 분석 실행
batch_size = 32
scores = []

for i in tqdm(range(0, len(df), batch_size)):
    batch_texts = df['comment'].iloc[i:i+batch_size].tolist()
    try:
        probs = analyze_sentiment_batch(batch_texts)
        for prob in probs:
            pos = prob[2]
            neg = prob[0]
            neu = prob[1]
            # compound 계산식: (pos - neg) * (1 - neu)
            compound = round((pos - neg) * (1 - neu), 4)
            scores.append(compound)
    except Exception as e:
        print(f"❌ 오류 발생 at batch {i}: {e}")
        scores.extend([0.0] * len(batch_texts))

# ✅ 결과 저장
df['sentiment_score'] = scores
output_file = filename.replace(".xlsx", "_roberta_sentiment_score_batch.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)

print("✅ 배치 기반 RoBERTa 감성 분석 완료! 정확하고 빠르게 처리됐습니다.")

Saving GPT 샘플링 10000_BIG5,주관,양극,인간유사성.xlsx to GPT 샘플링 10000_BIG5,주관,양극,인간유사성.xlsx


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]


100%|██████████| 313/313 [00:33<00:00,  9.42it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ 배치 기반 RoBERTa 감성 분석 완료! 정확하고 빠르게 처리됐습니다.


#9. 회귀분석

In [ ]:
pip install pandas numpy scikit-learn openpyxl

X -> Y

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from google.colab import files

# ✅ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ✅ 엑셀 데이터 로딩
df = pd.read_excel(filename)

# ✅ 컬럼 확인
print("\n✅ 데이터셋 컬럼 목록:")
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

# ✅ 사용자 입력: 종속 변수
target_column = input("\n🎯 종속변수로 사용할 컬럼명을 입력하세요: ")

# ✅ 사용자 입력: 독립 변수
input_columns = input("\n📌 독립변수로 사용할 컬럼명을 쉼표(,)로 구분해서 입력하세요:\n👉 ")
independent_columns = [col.strip() for col in input_columns.split(",")]

# ✅ X, y 정의
X = df[independent_columns]
y = df[target_column]

# ✅ 공분산 행렬 출력
print("\n📊 [공분산 행렬] (독립변수 간 상관 정도)")
print(X.cov())

# ✅ 상관계수 행렬 출력
print("\n📈 [상관계수 행렬] (독립변수 간 상관 정도)")
print(X.corr())

# ✅ 상관계수가 0.7 이상인 변수쌍만 추출
print("\n📌 [상관계수 ≥ 0.7 인 변수쌍 목록]")
corr_matrix = X.corr()
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        var1 = corr_matrix.columns[i]
        var2 = corr_matrix.columns[j]
        corr_value = corr_matrix.iloc[i, j]
        if abs(corr_value) >= 0.7:
            high_corr_pairs.append((var1, var2, corr_value))

if high_corr_pairs:
    for var1, var2, corr in high_corr_pairs:
        print(f"{var1} ↔ {var2}: 상관계수 = {corr:.3f}")
else:
    print("✅ 0.7 이상인 변수쌍 없음")

# ✅ VIF 계산
print("\n📌 [VIF (분산팽창계수) 결과]")
X_vif = sm.add_constant(X)  # 절편 포함
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data)

# ✅ 회귀 분석 실행
model = sm.OLS(y, X_vif).fit()

# ✅ 회귀 분석 요약 출력
print("\n📈 [회귀분석 결과 요약]")
print(model.summary())

# ✅ 유의성 표시용 별 생성 함수
def pval_to_stars(p):
    if p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return ""

# ✅ 회귀 계수 요약표 생성
summary_df = pd.DataFrame({
    "Variable": model.params.index,
    "Coefficient": model.params.values,
    "P-value": model.pvalues.values,
    "Significance": model.pvalues.apply(pval_to_stars)
})

# ✅ 출력
print("\n📋 회귀 계수 + 유의성(* 표시) 요약표:")
print(summary_df.to_string(index=False))
print(f"\n📌 Adjusted R² (조정된 설명력): {model.rsquared_adj:.4f}")


Saving 최종데이터_조절변수.xlsx to 최종데이터_조절변수 (2).xlsx

✅ 데이터셋 컬럼 목록:
0: Review_ID
1: User_Name
2: Rating
3: Helpful_Vote
4: Date
5: Review_Raw
6: Review
7: topic_id
8: meta_topic
9: review_length
10: LearningSupport_score
11: ChatbotInteraction_score
12: AppExperience_score
13: FeatureRequest_score
14: BiasConcerns_score
15: Humanlike_score
16: Subjectivity_score
17: Polarity_Score
18: sentiment_score
19: review_length_MC
20: LearningSupport_score_MC
21: ChatbotInteraction_score_MC
22: AppExperience_score_MC
23: FeatureRequest_score_MC
24: Bias_concerns_score_MC
25: Humanlike_score_MC
26: Subjectivity_score_MC
27: Polarity_Score_MC
28: review_length*Humanlike_MC
29: LearningSupport*Humanlike_MC
30: ChatbotInteraction*Humanlike_MC
31: AppExperience*Humanlike_MC
32: FeatureRequest*Humanlike_MC
33: Bias_concerns*Humanlike_MC
34: Subjectivity*Humanlike_MC
35: Polarity*Humanlike_MC

🎯 종속변수로 사용할 컬럼명을 입력하세요: sentiment_score

📌 독립변수로 사용할 컬럼명을 쉼표(,)로 구분해서 입력하세요:
👉 LearningSupport_score,ChatbotInteracti

표준화 베타값 추가

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from google.colab import files

# ✅ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ✅ 엑셀 데이터 로딩
df = pd.read_excel(filename)

# ✅ 컬럼 확인
print("\n✅ 데이터셋 컬럼 목록:")
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

# ✅ 사용자 입력: 종속 변수
target_column = input("\n🎯 종속변수로 사용할 컬럼명을 입력하세요: ")

# ✅ 사용자 입력: 독립 변수
input_columns = input("\n📌 독립변수로 사용할 컬럼명을 쉼표(,)로 구분해서 입력하세요:\n👉 ")
independent_columns = [col.strip() for col in input_columns.split(",")]

# ✅ X, y 정의
X = df[independent_columns]
y = df[target_column]

# ✅ 공분산 행렬 출력
print("\n📊 [공분산 행렬] (독립변수 간 상관 정도)")
print(X.cov())

# ✅ 상관계수 행렬 출력
print("\n📈 [상관계수 행렬] (독립변수 간 상관 정도)")
print(X.corr())

# ✅ 상관계수가 0.7 이상인 변수쌍만 추출
print("\n📌 [상관계수 ≥ 0.7 인 변수쌍 목록]")
corr_matrix = X.corr()
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        var1 = corr_matrix.columns[i]
        var2 = corr_matrix.columns[j]
        corr_value = corr_matrix.iloc[i, j]
        if abs(corr_value) >= 0.7:
            high_corr_pairs.append((var1, var2, corr_value))

if high_corr_pairs:
    for var1, var2, corr in high_corr_pairs:
        print(f"{var1} ↔ {var2}: 상관계수 = {corr:.3f}")
else:
    print("✅ 0.7 이상인 변수쌍 없음")

# ✅ VIF 계산
print("\n📌 [VIF (분산팽창계수) 결과]")
X_vif = sm.add_constant(X)  # 절편 포함
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data)

# ✅ 회귀 분석 실행 (비표준화)
model = sm.OLS(y, X_vif).fit()

# ✅ 표준화된 베타 계산용 모델 (표준화 후)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten()

X_scaled_const = sm.add_constant(X_scaled)
model_std = sm.OLS(y_scaled, X_scaled_const).fit()

# ✅ 유의성 표시용 별 생성 함수
def pval_to_stars(p):
    if p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return ""

# ✅ 회귀 계수 요약표 생성 (소수점 셋째 자리까지 반올림)
summary_df = pd.DataFrame({
    "Variable": model.params.index,
    "Coefficient": np.round(model.params.values, 3),
    "P-value": np.round(model.pvalues.values, 3),
    "Significance": model.pvalues.apply(pval_to_stars),
    "Standardized Beta": np.round(model_std.params, 3)  # <- 여기만 수정
})


# ✅ 출력
print("\n📋 회귀 계수 + 유의성(* 표시) + 표준화 베타 요약표:")
print(summary_df.to_string(index=False))
print(f"\n📌 Adjusted R² (조정된 설명력): {model.rsquared_adj:.4f}")


Saving 최종데이터_조절변수.xlsx to 최종데이터_조절변수 (1).xlsx

✅ 데이터셋 컬럼 목록:
0: Review_ID
1: User_Name
2: Rating
3: Helpful_Vote
4: Date
5: Review_Raw
6: Review
7: topic_id
8: meta_topic
9: review_length
10: LearningSupport_score
11: ChatbotInteraction_score
12: AppExperience_score
13: FeatureRequest_score
14: BiasConcerns_score
15: Humanlike_score
16: Subjectivity_score
17: Polarity_Score
18: sentiment_score
19: review_length_MC
20: LearningSupport_score_MC
21: ChatbotInteraction_score_MC
22: AppExperience_score_MC
23: FeatureRequest_score_MC
24: Bias_concerns_score_MC
25: Humanlike_score_MC
26: Subjectivity_score_MC
27: Polarity_Score_MC
28: review_length*Humanlike_MC
29: LearningSupport*Humanlike_MC
30: ChatbotInteraction*Humanlike_MC
31: AppExperience*Humanlike_MC
32: FeatureRequest*Humanlike_MC
33: Bias_concerns*Humanlike_MC
34: Subjectivity*Humanlike_MC
35: Polarity*Humanlike_MC

🎯 종속변수로 사용할 컬럼명을 입력하세요: sentiment_score

📌 독립변수로 사용할 컬럼명을 쉼표(,)로 구분해서 입력하세요:
👉 LearningSupport*Humanlike_MC,ChatbotIn

In [ ]:
# ✅ 결과들을 엑셀로 저장
from google.colab import files

# 📌 상관계수 0.7 이상 변수쌍 DataFrame 생성
if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs, columns=["변수1", "변수2", "상관계수"])
else:
    high_corr_df = pd.DataFrame([["없음", "없음", "N/A"]], columns=["변수1", "변수2", "상관계수"])

# 📌 회귀분석 요약 텍스트 저장용
model_summary_text = model.summary().as_text()
model_summary_df = pd.DataFrame(model_summary_text.splitlines(), columns=["회귀분석 요약"])

# ✅ ExcelWriter 사용하여 시트별 저장
output_filename = "회귀분석_전체결과.xlsx"
with pd.ExcelWriter(output_filename) as writer:
    X.cov().to_excel(writer, sheet_name="공분산 행렬")
    X.corr().to_excel(writer, sheet_name="상관계수 행렬")
    high_corr_df.to_excel(writer, sheet_name="상관 높은 변수쌍", index=False)
    vif_data.to_excel(writer, sheet_name="VIF 결과", index=False)
    summary_df.to_excel(writer, sheet_name="회귀 계수 요약", index=False)
    model_summary_df.to_excel(writer, sheet_name="회귀분석 텍스트", index=False)

# ✅ 파일 다운로드
files.download(output_filename)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

X -> M -> Y

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from google.colab import files
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ✅ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ✅ 엑셀 데이터 로딩
df = pd.read_excel(filename)

# ✅ 컬럼 목록 출력
print("\n✅ 데이터셋 컬럼 목록:")
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

# ✅ 사용자 입력
target_column = input("\n🎯 종속변수로 사용할 컬럼명을 입력하세요: ")
input_columns = input("\n📌 독립변수로 사용할 컬럼명을 쉼표(,)로 구분해서 입력하세요:\n👉 ")
independent_columns = [col.strip() for col in input_columns.split(",")]
mediator_column = input("\n🧩 매개변수(중재변수)로 사용할 컬럼명을 입력하세요:\n👉 ")

# ✅ 변수 설정
X = df[independent_columns]
M = df[[mediator_column]]
y = df[target_column]

# ✅ 공분산 행렬 출력
print("\n📊 [공분산 행렬]")
print(pd.concat([X, M, y], axis=1).cov())

# ✅ 표준화 함수
def standardize(df):
    return pd.DataFrame(StandardScaler().fit_transform(df), columns=df.columns)

# ✅ 유의성 표시 함수
def significance_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    elif p < 0.1:
        return '.'
    else:
        return ''

def print_coef_with_significance(model):
    summary_df = pd.DataFrame({
        'Coef': model.params,
        'p-value': model.pvalues,
        'Signif': model.pvalues.apply(significance_stars)
    })
    print(summary_df)

# ✅ 표준화된 데이터
X_std = standardize(X)
M_std = standardize(M)
y_std = standardize(y.to_frame())

# ✅ 1단계: X ➝ y
X1_const = sm.add_constant(X)
model1 = sm.OLS(y, X1_const).fit()

X1_std_const = sm.add_constant(X_std)
model1_std = sm.OLS(y_std, X1_std_const).fit()

print("\n🔹 [1단계: 독립변수 ➝ 종속변수]")
print(model1.summary())
print(f"\n📌 Adjusted R²: {model1.rsquared_adj:.4f}")
print("\n📊 Standardized Betas with significance:")
print_coef_with_significance(model1_std)

# ✅ 2단계: X ➝ M
model2 = sm.OLS(M, X1_const).fit()
model2_std = sm.OLS(M_std, X1_std_const).fit()

print("\n🔹 [2단계: 독립변수 ➝ 매개변수]")
print(model2.summary())
print(f"\n📌 Adjusted R²: {model2.rsquared_adj:.4f}")
print("\n📊 Standardized Betas with significance:")
print_coef_with_significance(model2_std)

# ✅ 3단계: X + M ➝ y
XM = pd.concat([X, M], axis=1)
XM_const = sm.add_constant(XM)
model3 = sm.OLS(y, XM_const).fit()

XM_std = pd.concat([X_std, M_std], axis=1)
XM_std_const = sm.add_constant(XM_std)
model3_std = sm.OLS(y_std, XM_std_const).fit()

print("\n🔹 [3단계: 독립변수 + 매개변수 ➝ 종속변수]")
print(model3.summary())
print(f"\n📌 Adjusted R²: {model3.rsquared_adj:.4f}")
print("\n📊 Standardized Betas with significance:")
print_coef_with_significance(model3_std)

# ✅ VIF 계산 함수
def compute_vif(df_with_const):
    vif_df = pd.DataFrame()
    vif_df["Variable"] = df_with_const.columns
    vif_df["VIF"] = [variance_inflation_factor(df_with_const.values, i)
                     for i in range(df_with_const.shape[1])]
    return vif_df

# ✅ VIF 출력
print("\n📌 [VIF (다중공선성 지표)]")
vif_results = compute_vif(XM_const)
print(vif_results)

# ✅ 위계회귀 검정 (R² 증가량의 유의성 검정)
anova_results = sm.stats.anova_lm(model1, model3)

anova_results_rounded = anova_results.copy()
anova_results_rounded['F'] = anova_results_rounded['F'].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "")
anova_results_rounded['Pr(>F)'] = anova_results_rounded['Pr(>F)'].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "")

print("\n📈 [위계회귀: R² 증가 유의성 검정 (ANOVA 비교)]")
print(anova_results_rounded)


Saving GPT 최종데이터_ver5_조절변수측정용.xlsx to GPT 최종데이터_ver5_조절변수측정용 (1).xlsx

✅ 데이터셋 컬럼 목록:
0: reviewId
1: 사용자 이름
2: 좋아요 개수
3: 작성 날짜
4: comment
5: Openness_score
6: Conscientiousness_score
7: Extraversion_score
8: Agreeableness_score
9: Neuroticism_score
10: review_length
11: Subjectivity_score
12: Certainty_score
13: human likeness
14: sentiment_score
15: star rating
16: Openness_score_MC
17: Conscientiousness_score_MC
18: Extraversion_score_MC
19: Agreeableness_score_MC
20: Neuroticism_score_MC
21: review_length_MC
22: Subjectivity_score_MC
23: Certainty_score_MC
24: human likeness_MC
25: Openness_score * human likeness_MC
26: Conscientiousness_score * human likeness_MC
27: Extraversion_score * human likeness_MC
28: Agreeableness_score * human likeness_MC
29: Neuroticism_score * human likeness_MC
30: review_length * human likeness_MC
31: Subjectivity_score * human likeness_MC
32: Certainty_score * human likeness_MC

🎯 종속변수로 사용할 컬럼명을 입력하세요: star rating

📌 독립변수로 사용할 컬럼명을 쉼표(,)로 구분해서 입력하세요:
👉 

NameError: name 'XM_const' is not defined

2단계 3단계 VIF 값 추가

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from google.colab import files
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ✅ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ✅ 엑셀 데이터 로딩
df = pd.read_excel(filename)

# ✅ 컬럼 목록 출력
print("\n✅ 데이터셋 컬럼 목록:")
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

# ✅ 사용자 입력
target_column = input("\n🎯 종속변수로 사용할 컬럼명을 입력하세요: ")
input_columns = input("\n📌 독립변수로 사용할 컬럼명을 쉼표(,)로 구분해서 입력하세요:\n👉 ")
independent_columns = [col.strip() for col in input_columns.split(",")]
mediator_column = input("\n🧩 매개변수(중재변수)로 사용할 컬럼명을 입력하세요:\n👉 ")

# ✅ 변수 설정
X = df[independent_columns]
M = df[[mediator_column]]
y = df[target_column]

# ✅ 공분산 행렬 출력
print("\n📊 [공분산 행렬]")
print(pd.concat([X, M, y], axis=1).cov())

# ✅ 표준화 함수
def standardize(df):
    return pd.DataFrame(StandardScaler().fit_transform(df), columns=df.columns)

# ✅ 유의성 표시 함수
def significance_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    elif p < 0.1:
        return '.'
    else:
        return ''

def print_coef_with_significance(model):
    summary_df = pd.DataFrame({
        'Coef': model.params,
        'p-value': model.pvalues,
        'Signif': model.pvalues.apply(significance_stars)
    })
    print(summary_df)

# ✅ 표준화된 데이터
X_std = standardize(X)
M_std = standardize(M)
y_std = standardize(y.to_frame())

# ✅ 1단계: X ➝ y
X1_const = sm.add_constant(X)
model1 = sm.OLS(y, X1_const).fit()

X1_std_const = sm.add_constant(X_std)
model1_std = sm.OLS(y_std, X1_std_const).fit()

print("\n🔹 [1단계: 독립변수 ➝ 종속변수]")
print(model1.summary())
print(f"\n📌 Adjusted R²: {model1.rsquared_adj:.4f}")
print("\n📊 Standardized Betas with significance:")
print_coef_with_significance(model1_std)

# ✅ 2단계: X ➝ M
model2 = sm.OLS(M, X1_const).fit()
model2_std = sm.OLS(M_std, X1_std_const).fit()

print("\n🔹 [2단계: 독립변수 ➝ 매개변수]")
print(model2.summary())
print(f"\n📌 Adjusted R²: {model2.rsquared_adj:.4f}")
print("\n📊 Standardized Betas with significance:")
print_coef_with_significance(model2_std)

# ✅ 2단계 VIF 출력
def compute_vif(df_with_const):
    vif_df = pd.DataFrame()
    vif_df["Variable"] = df_with_const.columns
    vif_df["VIF"] = [variance_inflation_factor(df_with_const.values, i)
                     for i in range(df_with_const.shape[1])]
    return vif_df

print("\n📌 [2단계 VIF (다중공선성 지표)]")
vif_results_2nd = compute_vif(X1_const)
print(vif_results_2nd)

# ✅ 3단계: X + M ➝ y
XM = pd.concat([X, M], axis=1)
XM_const = sm.add_constant(XM)
model3 = sm.OLS(y, XM_const).fit()

XM_std = pd.concat([X_std, M_std], axis=1)
XM_std_const = sm.add_constant(XM_std)
model3_std = sm.OLS(y_std, XM_std_const).fit()

print("\n🔹 [3단계: 독립변수 + 매개변수 ➝ 종속변수]")
print(model3.summary())
print(f"\n📌 Adjusted R²: {model3.rsquared_adj:.4f}")
print("\n📊 Standardized Betas with significance:")
print_coef_with_significance(model3_std)

# ✅ 3단계 VIF 출력
print("\n📌 [VIF (다중공선성 지표)]")
vif_results = compute_vif(XM_const)
print(vif_results)

# ✅ 위계회귀 검정 (R² 증가량의 유의성 검정)
anova_results = sm.stats.anova_lm(model1, model3)

anova_results_rounded = anova_results.copy()
anova_results_rounded['F'] = anova_results_rounded['F'].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "")
anova_results_rounded['Pr(>F)'] = anova_results_rounded['Pr(>F)'].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "")

print("\n📈 [위계회귀: R² 증가 유의성 검정 (ANOVA 비교)]")
print(anova_results_rounded)


Saving 최종데이터_조절변수.xlsx to 최종데이터_조절변수.xlsx

✅ 데이터셋 컬럼 목록:
0: Review_ID
1: User_Name
2: Rating
3: Helpful_Vote
4: Date
5: Review_Raw
6: Review
7: topic_id
8: meta_topic
9: review_length
10: LearningSupport_score
11: ChatbotInteraction_score
12: AppExperience_score
13: FeatureRequest_score
14: BiasConcerns_score
15: Humanlike_score
16: Subjectivity_score
17: Polarity_Score
18: sentiment_score
19: review_length_MC
20: LearningSupport_score_MC
21: ChatbotInteraction_score_MC
22: AppExperience_score_MC
23: FeatureRequest_score_MC
24: Bias_concerns_score_MC
25: Humanlike_score_MC
26: Subjectivity_score_MC
27: Polarity_Score_MC
28: review_length*Humanlike_MC
29: LearningSupport*Humanlike_MC
30: ChatbotInteraction*Humanlike_MC
31: AppExperience*Humanlike_MC
32: FeatureRequest*Humanlike_MC
33: Bias_concerns*Humanlike_MC
34: Subjectivity*Humanlike_MC
35: Polarity*Humanlike_MC

🎯 종속변수로 사용할 컬럼명을 입력하세요: sentiment_score

📌 독립변수로 사용할 컬럼명을 쉼표(,)로 구분해서 입력하세요:
👉 LearningSupport_score,ChatbotInteraction_s

In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 5.0 MB/s eta 0:00:00


In [ ]:
# ✅ 표준화된 회귀계수 출력 함수 (DataFrame 반환)
def print_coef_with_significance_df(model):
    return pd.DataFrame({
        'Coef': model.params,
        'p-value': model.pvalues,
        'Significance': model.pvalues.apply(significance_stars)
    })

# ✅ 출력용 객체 정리
output_dict = {
    "Covariance Matrix": pd.concat([X, M, y], axis=1).cov(),
    "Step1_Standardized": print_coef_with_significance_df(model1_std),
    "Step2_Standardized": print_coef_with_significance_df(model2_std),
    "Step3_Standardized": print_coef_with_significance_df(model3_std),
    "Hierarchical_Regression_ANOVA": anova_results
}

output_filename = "mediation_analysis_results.xlsx"

import xlsxwriter  # 설치 안 되어 있으면 '!pip install xlsxwriter' 실행 필요

with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
    # ✅ 공분산 행렬 저장
    output_dict["Covariance Matrix"].to_excel(writer, sheet_name="Covariance Matrix")

    # ✅ 1단계 summary 텍스트 저장
    worksheet = writer.book.add_worksheet('Step1_Summary')
    writer.sheets['Step1_Summary'] = worksheet
    for i, line in enumerate(model1.summary().as_text().split('\n')):
        worksheet.write(i, 0, line)

    # ✅ 2단계 summary 텍스트 저장
    worksheet = writer.book.add_worksheet('Step2_Summary')
    writer.sheets['Step2_Summary'] = worksheet
    for i, line in enumerate(model2.summary().as_text().split('\n')):
        worksheet.write(i, 0, line)

    # ✅ 3단계 summary 텍스트 저장
    worksheet = writer.book.add_worksheet('Step3_Summary')
    writer.sheets['Step3_Summary'] = worksheet
    for i, line in enumerate(model3.summary().as_text().split('\n')):
        worksheet.write(i, 0, line)

    # ✅ 표준화 회귀계수 저장
    output_dict["Step1_Standardized"].to_excel(writer, sheet_name="Step1_Std_Betas")
    output_dict["Step2_Standardized"].to_excel(writer, sheet_name="Step2_Std_Betas")
    output_dict["Step3_Standardized"].to_excel(writer, sheet_name="Step3_Std_Betas")

    # ✅ 위계회귀 결과 저장
    output_dict["Hierarchical_Regression_ANOVA"].to_excel(writer, sheet_name="Hierarchical_ANOVA")

# ✅ 파일 다운로드
from google.colab import files
files.download(output_filename)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>